Scans the full nested archive recursively, extracts all files for station 2170, parses PQPREVI time series, and plots model forecasts at lead times 6h, 12h, 24h, 48h.

In [2]:
from __future__ import annotations
import re, zipfile
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

ZIP_PATH = Path(r"C:\Users\simon\Documents\DP\analysis\data\Data Fornisseurs\OFEV_model\swisstransfer_e4eb4e12-cd87-4d66-be08-4bd5ea69dadf.zip")
STATION_ID = "2170"
LEAD_TIMES = [6, 12, 24, 36, 48]
OUT_DIR = Path("../outputs/OFEV_probabilistic").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Regexes for parsing
ROW_RE = re.compile(r"^\s*(\d{2})\s+(\d{2})\s+(\d{4})\s+(\d{2})\s+([-+]?\d+(?:[.,]\d+)?)\s+([-+]?\d+(?:[.,]\d+)?)\s*$")
NUM_RE = re.compile(r"^[-+]?\d+(?:[.,]\d+)?$")

# Functions for processing the nested zip files, recursively extracts all files from nested zip archives and parsing the tables inside them
def iter_nested_files(zf: zipfile.ZipFile, prefix: str = ""):
    for info in zf.infolist():
        if info.is_dir():
            continue
        name, raw = info.filename, zf.read(info)
        vpath = f"{prefix}{name}"
        if name.lower().endswith(".zip"):
            try:
                with zipfile.ZipFile(BytesIO(raw)) as nested:
                    yield from iter_nested_files(nested, vpath + "::")
            except zipfile.BadZipFile:
                pass
        else:
            yield vpath, raw


def model_from_filename(vpath: str) -> str | None:
    name = Path(vpath.split("::")[-1]).name
    m = re.match(rf"Pqprevi_(.+?)_{STATION_ID}(?:\.[A-Za-z0-9]+)?$", name)
    return m.group(1) if m else None

# Extract issue time from file content or fallback to filename timestamp
def parse_issue_time(vpath: str, text: str) -> pd.Timestamp | pd.NaT:
    m = re.search(r"(?:Emission:|Ausgegeben am.*?:)\s*(\d{1,2})\.(\d{1,2})\.(\d{4}),\s*(\d{1,2})[.:](\d{2})", text)
    if m:
        d, mo, y, hh, mm = m.groups()
        return pd.Timestamp(f"{y}-{int(mo):02d}-{int(d):02d} {int(hh):02d}:{mm}:00")
    m = re.search(r"(20\d{2})(\d{2})(\d{2})[-_]?(\d{2})(\d{2})", vpath) or re.search(r"(20\d{2})(\d{2})(\d{2})(\d{2})", vpath)
    if not m:
        return pd.NaT
    y, mo, d, hh = m.groups()[:4]
    mm = m.groups()[4] if len(m.groups()) > 4 else "00"
    return pd.Timestamp(f"{y}-{mo}-{d} {hh}:{mm}:00")


def _f(x: str) -> float:
    try:
        return float(str(x).replace(",", "."))
    except Exception:
        return np.nan


def _pick(cols: list[str], *keys: str) -> str | None:
    cl = {c.lower(): c for c in cols}
    for k in keys:
        for lk, c in cl.items():
            if k in lk:
                return c
    return None

# Read tables from a single FOEN file content, handling both deterministic and ensemble formats
def read_tables(vpath: str, text: str, model: str, issue: pd.Timestamp):
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    if not lines:
        return [], []

    i = next((k for k, ln in enumerate(lines) if re.search(r"\bdd\s+mm\s+yyyy\s+hh\b", ln, re.I)), None)
    if i is None:
        rows = []
        for ln in lines:
            m = ROW_RE.match(ln)
            if not m:
                continue
            dd, mm, yyyy, hh, h, q = m.groups()
            vt = pd.Timestamp(f"{yyyy}-{mm}-{dd} {hh}:00:00")
            lt = int(round((vt - issue).total_seconds() / 3600))
            if lt in LEAD_TIMES and lt >= 0:
                rows.append({"model": model, "issue_time": issue, "valid_time": vt, "lead_time_h": lt, "H": _f(h), "Q": _f(q), "source_path": vpath})
        return rows, []

    head = re.split(r"\s+", lines[i])
    if len(head) < 6:
        return [], []
    base, cols = ["dd", "mm", "yyyy", "hh"], head[4:]

    # Ensemble files expose columns like H_ctl/H_e01... and Q_ctl/Q_e01... + Q_min..Q_max.
    is_ensemble = any(re.fullmatch(r"[HQ]_(ctl|e\d{2}|min|p25|p50|p75|max)", c, flags=re.I) for c in cols)
    h_col = _pick(cols, "wasserstand", "niveau", "h")
    q_col = _pick(cols, "abfluss", "debit", "dbit", "débit", "débits", "q")

    det, ens = [], []
    for ln in lines[i + 1 :]:
        toks = re.split(r"\s+", ln)
        if len(toks) < 4 + len(cols) or not all(NUM_RE.match(t) for t in toks[:4]):
            continue
        row = dict(zip(base + cols, toks[: 4 + len(cols)]))
        vt = pd.Timestamp(f"{row['yyyy']}-{row['mm']}-{row['dd']} {row['hh']}:00:00")
        lt = int(round((vt - issue).total_seconds() / 3600))
        if lt not in LEAD_TIMES or lt < 0:
            continue

        if is_ensemble:
            ens.append({
                "model": model,
                "issue_time": issue,
                "valid_time": vt,
                "lead_time_h": lt,
                "source_path": vpath,
                **{c: _f(row.get(c, np.nan)) for c in cols},
            })
        else:
            det.append({
                "model": model,
                "issue_time": issue,
                "valid_time": vt,
                "lead_time_h": lt,
                "H": _f(row.get(h_col, np.nan)) if h_col else np.nan,
                "Q": _f(row.get(q_col, np.nan)) if q_col else np.nan,
                "source_path": vpath,
            })
    return det, ens


def _df(rows: list[dict], cols: list[str]) -> pd.DataFrame:
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=cols)

# Main processing loop: iterate all files in the nested zip, filter for station 2170 deterministic and ensemble files, parse tables and accumulate rows
det_rows, ens_rows = [], []
with zipfile.ZipFile(ZIP_PATH) as zf:
    for vpath, raw in iter_nested_files(zf):
        name = Path(vpath.split("::")[-1]).name
        if f"_{STATION_ID}" not in name or "Pqprevi_" not in name:
            continue
        model = model_from_filename(vpath)
        if not model:
            continue
        text = raw.decode("utf-8", errors="ignore")
        issue = parse_issue_time(vpath, text)
        if pd.isna(issue):
            continue
        d, e = read_tables(vpath, text, model, issue)
        det_rows.extend(d)
        ens_rows.extend(e)

if not det_rows and not ens_rows:
    raise RuntimeError(f"No station {STATION_ID} data found")

# Deterministic DF (files with simple H/Q columns)
df_det = _df(det_rows, ["model", "issue_time", "valid_time", "lead_time_h", "H", "Q", "source_path"])\
    .sort_values(["lead_time_h", "issue_time", "model"], na_position="last")\
    .reset_index(drop=True)

# Ensemble/member DF (rows with columns like H_ctl...Q_max)
df_ens = _df(ens_rows, ["model", "issue_time", "valid_time", "lead_time_h", "source_path"])\
    .sort_values(["lead_time_h", "issue_time", "model"], na_position="last")\
    .reset_index(drop=True)

# Per-model Q quantiles DF
q_cols = [c for c in ["Q_min", "Q_p25", "Q_p50", "Q_p75", "Q_max"] if c in df_ens.columns]
df_q = (
    df_ens[["model", "issue_time", "valid_time", "lead_time_h", *q_cols]]
    .groupby(["model", "issue_time", "valid_time", "lead_time_h"], as_index=False)
    .median(numeric_only=True)
    .sort_values(["lead_time_h", "model", "issue_time"], na_position="last")
    if q_cols else pd.DataFrame(columns=["model", "issue_time", "valid_time", "lead_time_h", "Q_min", "Q_p25", "Q_p50", "Q_p75", "Q_max"])
)


print("deterministic rows:", len(df_det))
print("ensemble rows:", len(df_ens))
print("quantile rows:", len(df_q))
print("saved to:", OUT_DIR)

deterministic rows: 64849
ensemble rows: 19714
quantile rows: 18162
saved to: C:\Users\simon\Documents\DP\analysis\outputs\OFEV_probabilistic


In [4]:
# display samples
display(df_det.head())
display(df_ens.head())
display(df_q.head())

,model,issue_time,valid_time,lead_time_h,H,Q,source_path
0,COSMO1,2020-06-11 06:24:00,2020-06-11 12:00:00,6,379.94,114.4,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
1,COSMO7,2020-06-11 06:24:00,2020-06-11 12:00:00,6,379.94,114.5,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
2,COSMOE_Con,2020-06-11 06:24:00,2020-06-11 12:00:00,6,379.94,114.4,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
3,COSMOE_Med,2020-06-11 06:24:00,2020-06-11 12:00:00,6,379.94,114.4,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...
4,ECMNOR,2020-06-11 06:24:00,2020-06-11 12:00:00,6,379.94,114.5,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...


,model,issue_time,valid_time,lead_time_h,source_path,H_e01,H_e02,H_e03,H_e04,H_e05,...,H_p50,H_p75,H_max,Q_min,Q_p25,Q_p50,Q_p75,Q_max,H_ctl,Q_ctl
0,COSMOE,2020-06-11 06:24:00,2020-06-11 12:00:00,6,2020.zip::2020/2020061107_pqprevi.zip::Pqprevi...,379.94,379.94,379.94,379.94,379.94,...,379.94,379.94,379.94,114.5,114.5,114.5,114.6,114.6,NaN,NaN
1,COSMOE,2020-06-12 06:58:00,2020-06-12 13:00:00,6,2020.zip::2020/2020061208_pqprevi.zip::Pqprevi...,379.87,379.86,379.86,379.86,379.87,...,379.86,379.87,379.90,103.9,104.0,104.1,104.6,108.5,NaN,NaN
2,COSMOE,2020-06-13 05:47:00,2020-06-13 12:00:00,6,2020.zip::2020/2020061307_pqprevi.zip::Pqprevi...,379.76,379.76,379.76,379.76,379.76,...,379.76,379.76,379.76,90.6,90.7,90.8,91.0,91.2,NaN,NaN
3,COSMOE,2020-06-13 10:48:00,2020-06-13 17:00:00,6,2020.zip::2020/2020061319_pqprevi.zip::Pqprevi...,379.70,379.67,379.70,379.66,379.76,...,379.69,379.72,379.77,78.9,79.4,82.0,86.1,92.2,NaN,NaN
4,COSMOE,2020-06-13 10:48:00,2020-06-13 17:00:00,6,2020.zip::2020/2020061321_pqprevi.zip::Pqprevi...,379.70,379.67,379.70,379.66,379.76,...,379.69,379.72,379.77,78.9,79.4,82.0,86.1,92.2,NaN,NaN


,model,issue_time,valid_time,lead_time_h,Q_min,Q_p25,Q_p50,Q_p75,Q_max
0,C1E,2020-09-15 06:33:00,2020-09-15 13:00:00,6,43.7,43.8,44.0,44.3,44.3
3,C1E,2020-09-15 17:15:00,2020-09-15 23:00:00,6,46.3,46.3,46.4,46.7,46.7
6,C1E,2020-09-16 06:04:00,2020-09-16 12:00:00,6,50.9,50.9,51.0,51.0,51.1
9,C1E,2020-09-17 06:11:00,2020-09-17 12:00:00,6,37.9,38.0,38.1,38.1,38.1
12,C1E,2020-09-18 06:01:00,2020-09-18 12:00:00,6,37.7,37.7,37.7,37.7,37.7


In [5]:
# Drop deterministic C1E rows with lead time 36, since very few data is available for that model and lead time.
TARGET_LT = 36
MODEL_SUBSTR = "C1E" 

def drop_ce1_lt36(df: pd.DataFrame, name: str) -> pd.DataFrame:
    lt = pd.to_numeric(df["lead_time_h"], errors="coerce").round().astype("Int64")
    is_target = lt.eq(TARGET_LT)
    is_model = df["model"].astype(str).str.contains(MODEL_SUBSTR, case=False, na=False)

    mask = is_target & is_model
    print(f"{name}: dropping {int(mask.sum())} rows")
    return df.loc[~mask].reset_index(drop=True)

df_det = drop_ce1_lt36(df_det, "df_det")
df_ens = drop_ce1_lt36(df_ens, "df_ens")
df_q   = drop_ce1_lt36(df_q,   "df_q")

df_det: dropping 46 rows
df_ens: dropping 23 rows
df_q: dropping 18 rows


In [6]:
# Export CSVs
df_det.to_csv(OUT_DIR / "station2170_deterministic.csv", index=False)
df_ens.to_csv(OUT_DIR / "station2170_ensemble_wide.csv", index=False)
df_q.to_csv(OUT_DIR / "station2170_q_quantiles_by_model.csv", index=False)

print("deterministic rows:", len(df_det))
print("ensemble rows:", len(df_ens))
print("quantile rows:", len(df_q))
print("saved to:", OUT_DIR)

deterministic rows: 64803
ensemble rows: 19691
quantile rows: 18144
saved to: C:\Users\simon\Documents\DP\analysis\outputs\OFEV_probabilistic
